# Time Series Analysis

In [35]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from eda_utils import line_with_slider
from plotly.subplots import make_subplots

from solar_predictor.data import drop_incomplete_tail, read_sarah3
from solar_predictor.metrics import calculate_metrics
from solar_predictor.storage import get_location_prefix
from solar_predictor.utils import LOCATIONS

px.defaults.height = 700

In [37]:
# skopje_lp = get_location_prefix(LOCATIONS["SKOPJE"]["LAT"], LOCATIONS["SKOPJE"]["LONG"])
# raw_df = read_sarah3(skopje_lp)
# raw_df.to_csv("raw_df.csv")

raw_df = pd.read_csv("raw_df.csv", parse_dates=["datetime"]).set_index("datetime")

In [38]:
df = drop_incomplete_tail(raw_df)
daily_df = df.resample("D").mean()

daily_df.tail()

,timestamp,shortwave_radiation,direct_radiation,diffuse_radiation
datetime,,,,
2026-10-26 00:00:00+00:00,1.793015e+09,48.625000,7.812500,40.812500
2026-10-27 00:00:00+00:00,1.793102e+09,49.979167,3.041667,46.937500
2026-10-28 00:00:00+00:00,1.793188e+09,44.375000,2.333333,42.041667
2026-10-29 00:00:00+00:00,1.793274e+09,61.770833,6.604167,55.166667
2026-10-30 00:00:00+00:00,1.793361e+09,49.958333,5.208333,44.750000


# Smoothing

## Simple Moving Average (SMA) and Exponential Moving Average (EMA)

In [39]:
windows = (5, 10, 30)

ma_df = daily_df[["shortwave_radiation"]].copy()
ma_df["prev"] = ma_df["shortwave_radiation"].shift(1)
metrics = []

for w in windows:
    ma_df[f"sma_{w}"] = ma_df["prev"].rolling(w).mean()
    ma_df[f"ema_{w}"] = ma_df["prev"].ewm(span=w, adjust=False, min_periods=w).mean()

    sma_metrics = calculate_metrics(ma_df["shortwave_radiation"], ma_df[f"sma_{w}"])
    metrics.append({"algorithm": "SMA", "w": w, **sma_metrics})
    ema_metrics = calculate_metrics(ma_df["shortwave_radiation"], ma_df[f"ema_{w}"])
    metrics.append({"algorithm": "EMA", "w": w, **ema_metrics})


colors = ["lightgray", *px.colors.qualitative.Plotly]
fig = make_subplots(
    rows=len(windows),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.02,
)
for row, w in enumerate(windows, start=1):
    columns = {
        "shortwave_radiation": "shortwave_radiation",
        "sma": f"sma_{w}",
        "ema": f"ema_{w}",
    }
    for (name, column), color in zip(columns.items(), colors):
        fig.add_trace(
            go.Scatter(
                x=ma_df.index,
                y=ma_df[column],
                name=name,
                line_color=color,
                legendgroup=name,
                showlegend=row == 1,
            ),
            row=row,
            col=1,
        )
    fig.update_yaxes(title_text=f"{w} days", row=row, col=1)

fig.update_xaxes(range=[pd.to_datetime("2007-01-01"), pd.to_datetime("2008-01-01")])
fig.update_xaxes(
    rangeslider={"visible": True, "thickness": 0.05}, row=len(windows), col=1
)
fig.update_layout(
    title="Shortwave Radiation Moving Averages by Window Size",
    height=800,
    legend={
        "orientation": "h",
        "y": 1.02,
        "x": 1,
        "xanchor": "right",
        "yanchor": "bottom",
    },
)
fig.show()

display(pd.DataFrame(metrics))

,algorithm,w,mse,rmse
0,SMA,5,3608.330132,60.069378
1,EMA,5,3254.113995,57.044842
2,SMA,10,3537.806914,59.479466
3,EMA,10,3273.990395,57.218794
4,SMA,30,3934.775290,62.727787
5,EMA,30,3744.561640,61.192823


Conclusion: Both methods (SMA and EMA) follow the same trend, and larger windows are smoother but react later.
The averages use only previous days, so they don't leak the predicted value.
Model validation error will show which method and window to use.

## Alpha in EMA

In [40]:
ema_df = daily_df[["shortwave_radiation"]].copy()
prev = ma_df["shortwave_radiation"].shift(1)


for alpha in [0.02, 0.08]:
    ema_df[f"ema_{alpha}"] = (
        prev.ewm(alpha=alpha, adjust=False).mean()
    )
    metrics.append(
        {
            "algorithm": "EMA",
            "alpha": alpha,
            **calculate_metrics(ema_df["shortwave_radiation"], ema_df[f"ema_{alpha}"]),
        }
    )


fig = line_with_slider(
    df=ema_df,
    window=pd.DateOffset(years=1),
    color_discrete_map={"shortwave_radiation": "lightgray"},
)
fig.show()

display(pd.DataFrame(metrics))

,algorithm,w,mse,rmse,alpha
0,SMA,5.0,3608.330132,60.069378,NaN
1,EMA,5.0,3254.113995,57.044842,NaN
2,SMA,10.0,3537.806914,59.479466,NaN
3,EMA,10.0,3273.990395,57.218794,NaN
4,SMA,30.0,3934.775290,62.727787,NaN
5,EMA,30.0,3744.561640,61.192823,NaN
6,EMA,NaN,6375.437913,79.846339,0.02
7,EMA,NaN,3565.986952,59.715885,0.08
